In [1]:
# import sys
# sys.path.append("..")

In [2]:
import pandas as pd

df_raw = pd.read_csv("../data/raw/questions.csv")
print(f"Shape: {df_raw.shape} \nColumns: {list(df_raw.columns)}")

df_raw.head()

Shape: (404351, 6) 
Columns: ['id', 'qid1', 'qid2', 'question1', 'question2', 'is_duplicate']


,id,qid1,qid2,question1,question2,is_duplicate
0,0,1,2,What is the step by step guide to invest in sh...,What is the step by step guide to invest in sh...,0
1,1,3,4,What is the story of Kohinoor (Koh-i-Noor) Dia...,What would happen if the Indian government sto...,0
2,2,5,6,How can I increase the speed of my internet co...,How can Internet speed be increased by hacking...,0
3,3,7,8,Why am I mentally very lonely? How can I solve...,Find the remainder when [math]23^{24}[/math] i...,0
4,4,9,10,"Which one dissolve in water quikly sugar, salt...",Which fish would survive in salt water?,0


# Drop and Missing Values
Since our data is not sequential (e.g. time series), we can drop the `id` column since it brings no analysis power.

Moreover, as we can see from the cell below the number of null values is negletable compared to the number of rows (~$0$%), hence they are dropped as well.

The small dataset containing the Null values is memorized in case is necessary. After dropping the rows the index is reseted, so the index doesn't contain gaps.

Finally, columns `qid1` and `qid2` are also dropped since the same exact question can be referenced to a different `qid`, this may result in a problem for the clustering algorithm.

In [3]:
a = df_raw[["qid1", "question1"]].set_axis(["qid", "text"], axis=1)
b = df_raw[["qid2", "question2"]].set_axis(["qid", "text"], axis=1)
long = pd.concat([a, b]).dropna()

n_ids = long.groupby("text").qid.nunique()
example = n_ids.sort_values(ascending=False).index[0] # e.g. the text with the most ids
print(long[long.text == example].drop_duplicates("qid").head(5))

          qid                                    text
14713   29368  What are the best ways to lose weight?
38208   76124  What are the best ways to lose weight?
56251  111914  What are the best ways to lose weight?
81378  161648  What are the best ways to lose weight?
81988  162850  What are the best ways to lose weight?


In [4]:
df = df_raw.drop(columns=["id", "qid1", "qid2"])

print("Number of Null Values Before")
print(df.isna().sum())


print(f"\nShape before: {df.shape}")
df_null = df[df.isna().any(axis=1)]

df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)

print("Number of Null Values After")
print(f"Shape after: {df.shape}\n")
print(df.isna().sum())
df_null.head()

Number of Null Values Before
question1       1
question2       2
is_duplicate    0
dtype: int64

Shape before: (404351, 3)
Number of Null Values After
Shape after: (404348, 3)

question1       0
question2       0
is_duplicate    0
dtype: int64


,question1,question2,is_duplicate
105796,How can I develop android app?,NaN,0
201871,How can I create an Android app?,NaN,0
363416,NaN,My Chinese name is Haichao Yu. What English na...,0


In [5]:
df.head()

,question1,question2,is_duplicate
0,What is the step by step guide to invest in sh...,What is the step by step guide to invest in sh...,0
1,What is the story of Kohinoor (Koh-i-Noor) Dia...,What would happen if the Indian government sto...,0
2,How can I increase the speed of my internet co...,How can Internet speed be increased by hacking...,0
3,Why am I mentally very lonely? How can I solve...,Find the remainder when [math]23^{24}[/math] i...,0
4,"Which one dissolve in water quikly sugar, salt...",Which fish would survive in salt water?,0


# Simple Baseline Check
Although accuracy won't be as good of a performance metric (F1 will be better), it's still good to establish some sort of a baseline. In this case, 63.07% will be our baseline for accuracy.

In [6]:
print(df.is_duplicate.value_counts())
baseline_acc = len(df[df.is_duplicate == 0]) / len(df)
print(f"Baseline Accuracy: {baseline_acc}")

is_duplicate
0    255042
1    149306
Name: count, dtype: int64
Baseline Accuracy: 0.6307487609682748


In [7]:
df["question1_norm"] = df["question1"]
df["question2_norm"] = df["question2"]

for col in ["question1_norm", "question2_norm"]:
    df[col] = df[col].str.normalize("NFKC")
    df[col] = df[col].str.lower()
    df[col] = df[col].str.split().str.join(" ")

target = df.pop("is_duplicate")
df["is_duplicate"] = target
df.head()

,question1,question2,question1_norm,question2_norm,is_duplicate
0,What is the step by step guide to invest in sh...,What is the step by step guide to invest in sh...,what is the step by step guide to invest in sh...,what is the step by step guide to invest in sh...,0
1,What is the story of Kohinoor (Koh-i-Noor) Dia...,What would happen if the Indian government sto...,what is the story of kohinoor (koh-i-noor) dia...,what would happen if the indian government sto...,0
2,How can I increase the speed of my internet co...,How can Internet speed be increased by hacking...,how can i increase the speed of my internet co...,how can internet speed be increased by hacking...,0
3,Why am I mentally very lonely? How can I solve...,Find the remainder when [math]23^{24}[/math] i...,why am i mentally very lonely? how can i solve...,find the remainder when [math]23^{24}[/math] i...,0
4,"Which one dissolve in water quikly sugar, salt...",Which fish would survive in salt water?,"which one dissolve in water quikly sugar, salt...",which fish would survive in salt water?,0


In [ ]:
# 1 Question ids: one integer per distinct text_norm, and pairs rewritten with those ids.
# 2 Pair checks: remove pairs where both sides are the same question, remove repeated pairs (smaller id first), and handle pairs that have both labels. Print each count.
# 3 Ground-truth groups: connected components of the pairs labelled duplicate, with the group size distribution.
# 4 Tuning and test split: about 20% / 80% of the questions, split by connected component of the whole pair graph and with a fixed seed, so that no question appears on both sides.
# 5 Save: questions.parquet (id, text_raw, text_norm, group, split), pairs.parquet (ids and is_duplicate), checked by reading them back.
# 6 Numbers for the report: class balance (36.9% duplicates), how many pairs each question appears in, and every count printed above.

404348